In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
df_orders = spark.read.format("delta").load(f"{SILVER_PATH}/orders")

In [0]:
df_order_items = spark.read.format("delta").load(f"{SILVER_PATH}/order_items")

In [0]:
df_customers = spark.read.format("delta").load(f"{SILVER_PATH}/customers")

In [0]:
df_payments = spark.read.format("delta").load(f"{SILVER_PATH}/payments")

In [0]:
df_shipments = spark.read.format("delta").load(f"{SILVER_PATH}/shipments")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.types import *

In [0]:
df_agg_order_items = df_order_items \
    .groupBy("order_id") \
    .agg(
        F.sum(F.col("quantity")).alias("total_quantity"),
        F.count("*").alias("order_item_count")
    )

In [0]:
df_orders_gold = df_orders.alias("o") \
    .join(
        df_customers.alias("c"),
        (
            (F.col("c.customer_id") == F.col("o.customer_id")) &
            (F.col("o.order_date") >= F.col("c.valid_from")) &
            (
                (F.col("o.order_date") < F.col("c.valid_to")) |
                F.col("valid_to").isNull()
            )
        ),
    ) \
    .join(
        df_agg_order_items.alias("oi"),
        "order_id"
    ) \
    .join(
        df_payments.alias("p"),
        "order_id"
    ) \
    .join(
        df_shipments.alias("s"),
        "order_id",
        "left"
    ) \
    .select(
        F.col("o.order_id"),
        F.col("c.customer_sk"),
        F.col("o.order_date"),
        F.col("o.order_status"),
        F.col("o.total_amount"),
        F.col("oi.total_quantity"),
        F.col("oi.order_item_count"),
        F.col("p.amount").alias("payment_amount"),
        F.col("p.payment_method"),
        F.col("p.payment_status"),
        F.col("p.payment_date"),
        F.col("s.shipment_status"),
        F.col("s.shipment_date"),
        F.col("s.delivery_date"),
        F.col("s.warehouse_id"),
        F.col("s.tracking_number")
    )

In [0]:
df_orders_gold.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/fact_orders")